# Long Short-Term Memory (LSTM) Model

This notebook demonstrates LSTM model training including:
- LSTM architecture and concepts
- Sequence creation with sliding windows
- Backpropagation Through Time (BPTT)
- Training and evaluation
- Performance analysis

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))

# Change to script directory
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(''))))

# Import training script
from training.train_lstm import main as train_lstm_main

print("LSTM training script imported successfully!")

## Step 1: LSTM Concepts Explained

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║           LSTM (LONG SHORT-TERM MEMORY) CELLS                     ║
╚════════════════════════════════════════════════════════════════════╝

LSTM Cell Components:

1. FORGET GATE (σ - sigmoid):
   f_t = σ(W_f · [h_(t-1), x_t] + b_f)
   - Controls which information from previous cell state to FORGET
   - Output: 0 (forget) to 1 (remember)

2. INPUT GATE (σ - sigmoid):
   i_t = σ(W_i · [h_(t-1), x_t] + b_i)
   - Controls which NEW information to ADD to cell state
   - Output: 0 (ignore) to 1 (accept)

3. CANDIDATE VALUES (tanh):
   C̃_t = tanh(W_c · [h_(t-1), x_t] + b_c)
   - Candidate values for adding to cell state
   - Output: -1 to 1

4. NEW CELL STATE:
   C_t = f_t * C_(t-1) + i_t * C̃_t
   - Combines: forget × old_state + input × candidates
   - C_(t-1): Cell state from previous timestep

5. OUTPUT GATE (σ - sigmoid):
   o_t = σ(W_o · [h_(t-1), x_t] + b_o)
   - Controls what information to output
   - Output: 0 (hide) to 1 (show)

6. HIDDEN STATE (output):
   h_t = o_t * tanh(C_t)
   - Output passed to next timestep
   - Also used for making predictions

═══════════════════════════════════════════════════════════════════════

ADVANTAGES OVER VANILLA RNN:

- SOLVES VANISHING GRADIENT PROBLEM:
  RNN: Gradients → 0 over many timesteps (can't learn long-term deps)
  LSTM: Cell state highway allows gradient flow (can learn 100+ timesteps)

- MEMORY MECHANISMS:
  Separate cell state (memory) from hidden state (output)
  Can maintain information across many timesteps

- GATING MECHANISMS:
  Learned gates control information flow
  Better selective learning than vanilla RNN
""")

## Step 2: Sequence Creation

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║          SLIDING WINDOW SEQUENCE CREATION                         ║
╚════════════════════════════════════════════════════════════════════╝

Original Data Shape: (392, 8)
  - 392 samples
  - 8 features (cylinders, displacement, horsepower, weight, ...)

Window Size: 10 timesteps

SLIDING WINDOW PROCESS:

Step 1: Create windows of 10 consecutive samples
        Each window covers 10 samples with 8 features

Step 2: Use the LAST sample's target as the label
        Predicting: y[i+9] from X[i:i+10]

Step 3: Create 3D tensor
        Shape: (383, 10, 8)
        - 383 sequences (392 - 10 + 1)
        - 10 timesteps per sequence
        - 8 features per timestep

EXAMPLE:
Sequence 0: samples 0-9   → predict y[9]
Sequence 1: samples 1-10  → predict y[10]
Sequence 2: samples 2-11  → predict y[11]
...
Sequence 382: samples 382-391 → predict y[391]

WHY SEQUENCES?
- LSTM cells process sequences step-by-step
- Each timestep uses previous hidden state
- Models temporal dependencies
- Can learn patterns like: weight → acceleration → efficiency
""")

## Step 3: Train LSTM Model

In [ ]:
# Execute the training script
model, metrics, training_time = train_lstm_main()

print(f"\nLSTM Model training completed!")
print(f"Training time: {training_time:.2f} seconds")

## Step 4: Model Architecture

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║            LSTM MODEL ARCHITECTURE                                ║
╚════════════════════════════════════════════════════════════════════╝

Input (3D Tensor)
  ├─ Samples: 383
  ├─ Timesteps: 10
  └─ Features: 8
       ↓
  LSTM Layer 1: 64 units
  - Input: (batch, 10, 8)
  - Output: (batch, 10, 64)  [return_sequences=True]
  - 64 LSTM cells process the sequence
  - Returns: full sequence of hidden states
       ↓
  Dropout(0.3): Stochastic regularization
  - Randomly deactivates 30% of units
  - Prevents co-adaptation of LSTM cells
       ↓
  LSTM Layer 2: 32 units
  - Input: (batch, 10, 64)
  - Output: (batch, 32)  [return_sequences=False]
  - 32 LSTM cells
  - Returns: only final hidden state
       ↓
  Dense Layer: 16 units + ReLU
  - Input: (batch, 32)
  - Output: (batch, 16)
  - Fully connected, ReLU activation
       ↓
  Output Dense Layer: 1 unit
  - Input: (batch, 16)
  - Output: (batch, 1) - MPG prediction
  - Linear activation for regression

═══════════════════════════════════════════════════════════════════════

TRAINING PROCESS (Backpropagation Through Time):

1. FORWARD PASS:
   - Input sequence moves through each timestep
   - LSTM cells update: forget gate, input gate, candidate values
   - Hidden state propagates forward in time

2. LOSS COMPUTATION:
   - MSE Loss = (y_true - y_pred)²
   - Gradient computed at final output

3. BACKWARD PASS (BPTT):
   - Backpropagation through entire sequence (all 10 timesteps)
   - Gradients computed for each LSTM gate
   - Gradients flow backward through time

4. WEIGHT UPDATES:
   - Adam optimizer updates all parameters
   - Adaptive learning rates per parameter
   - Learning rate: 0.001
""")

print("\nEvaluation Metrics:")
for key, value in metrics.items():
    if key != 'model_name':
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")

## Step 5: Visualizations

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display all generated plots
image_files = [
    'images/lstm/LSTM_training_history.png',
    'images/lstm/LSTM_predictions_vs_actual.png',
    'images/lstm/LSTM_residuals.png',
    'images/lstm/LSTM_error_distribution.png'
]

for img_path in image_files:
    if os.path.exists(img_path):
        img = Image.open(img_path)
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(img_path.split('/')[-1])
        plt.tight_layout()
        plt.show()
    else:
        print(f"Image not found: {img_path}")

## Step 6: LSTM vs MLP Comparison

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║     LSTM vs MLP: WHEN TO USE EACH                                 ║
╚════════════════════════════════════════════════════════════════════╝

MLP (Multilayer Perceptron):
✓ Simpler architecture, faster training
✓ Good for feature-based predictions (no time dimension)
✓ Lower memory requirements
✓ Lower computational complexity
✗ Cannot model sequential/temporal dependencies
✗ Treats all samples independently

LSTM (Long Short-Term Memory):
✓ Models temporal/sequential dependencies
✓ Can capture long-term patterns (100+ steps)
✓ Suitable for time series data
✓ Memory mechanisms for complex patterns
✗ More complex, slower training
✗ Higher memory requirements
✗ Risk of overfitting on small datasets

In this project:
- Auto MPG dataset is created as sequences for LSTM
- Sliding window simulates temporal dependencies
- LSTM captures relationships across 10-sample windows
- Useful for features that have temporal correlation
  (e.g., weight → horsepower → acceleration)
""")

## Step 7: Key Takeaways

In [ ]:
print(f"""
╔════════════════════════════════════════════════════════════════════╗
║              LSTM TRAINING SUMMARY                                ║
╚════════════════════════════════════════════════════════════════════╝

Model: LSTM-64 → Dropout → LSTM-32 → Dense-16 → Output
Training Time: {training_time:.2f} seconds

Performance Metrics:
  RMSE: {metrics.get('RMSE', 'N/A'):.4f}
  MAE: {metrics.get('MAE', 'N/A'):.4f}
  R²: {metrics.get('R2', 'N/A'):.4f}

Key Concepts demonstrated:
1. Sequence creation with sliding windows (10 timesteps)
2. LSTM cell architecture with gating mechanisms
3. Backpropagation Through Time (BPTT)
4. Memory preservation across timesteps
5. Vanishing gradient solution via LSTM gates
6. Stacked LSTM layers for increased capacity
7. Return sequences mechanism
8. Comprehensive evaluation and visualization
""")